# 3. The sixteen joints

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

The MANO right hand has 16 joints. Memorize the order as a table. The pinky is stored before the ring. The first joint of the thumb is the carpometacarpal joint.

Run the next cell. The printed tree and the table below are the same data.


In [1]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import mano_skeleton as sk

print(sk.format_tree())
print()
print("index 7:", sk.JOINT_NAMES[7], sk.ANATOMICAL[sk.JOINT_NAMES[7]])
print("index 10:", sk.JOINT_NAMES[10], sk.ANATOMICAL[sk.JOINT_NAMES[10]])
print("thumb1:", sk.ANATOMICAL["thumb1"])
print("parents:", sk.PARENTS.tolist())
print("fingertip vertices:", sk.TIP_VERTEX_IDS)
assert sk.JOINT_NAMES[7] == "pinky1"
assert sk.JOINT_NAMES[10] == "ring1"
assert int(sk.PARENTS[10]) == 0
assert sk.ANATOMICAL["thumb1"] == "thumb CMC"
assert len(sk.JOINT_NAMES) == 16


00 wrist    wrist        pose[00:03]  <- global rotation, not inside the 45-D finger pose
├─ 01 index1   index MCP    pose[03:06]
│  └─ 02 index2   index PIP    pose[06:09]
│     └─ 03 index3   index DIP    pose[09:12]
├─ 04 middle1  middle MCP   pose[12:15]
│  └─ 05 middle2  middle PIP   pose[15:18]
│     └─ 06 middle3  middle DIP   pose[18:21]
├─ 07 pinky1   pinky MCP    pose[21:24]
│  └─ 08 pinky2   pinky PIP    pose[24:27]
│     └─ 09 pinky3   pinky DIP    pose[27:30]
├─ 10 ring1    ring MCP     pose[30:33]
│  └─ 11 ring2    ring PIP     pose[33:36]
│     └─ 12 ring3    ring DIP     pose[36:39]
└─ 13 thumb1   thumb CMC    pose[39:42]
   └─ 14 thumb2   thumb MCP    pose[42:45]
      └─ 15 thumb3   thumb IP     pose[45:48]

index 7: pinky1 pinky MCP
index 10: ring1 ring MCP
thumb1: thumb CMC
parents: [-1, 0, 1, 2, 0, 4, 5, 0, 7, 8, 0, 10, 11, 0, 13, 14]
fingertip vertices: {'thumb': 744, 'index': 320, 'middle': 443, 'ring': 554, 'pinky': 671}


In [2]:
print(sk.JOINT_NAMES)

['wrist', 'index1', 'index2', 'index3', 'middle1', 'middle2', 'middle3', 'pinky1', 'pinky2', 'pinky3', 'ring1', 'ring2', 'ring3', 'thumb1', 'thumb2', 'thumb3']


All five fingers connect directly to the wrist. The ring finger's parent is the wrist, not the pinky. The pinky has a smaller index than the ring, and the two fingers are siblings in the tree.

MCP is the metacarpophalangeal joint, PIP the proximal interphalangeal joint, DIP the distal interphalangeal joint. The thumb has one fewer hinge: `thumb1` is the carpometacarpal joint CMC, `thumb2` is the metacarpophalangeal joint MCP, and `thumb3` is the interphalangeal joint IP.

## How the 48 numbers are sliced

The full pose has 48 numbers. Joint `i` occupies `[3i : 3i+3]`.

| Joints | Index | Slice of the full pose | Slice inside `hand_pose` |
|---|---|---|---|
| wrist | 0 | `[0:3]` | absent from `hand_pose`; this is `global_orient` |
| index1/2/3 | 1–3 | `[3:12]` | `[0:9]` |
| middle1/2/3 | 4–6 | `[12:21]` | `[9:18]` |
| pinky1/2/3 | 7–9 | `[21:30]` | `[18:27]` |
| ring1/2/3 | 10–12 | `[30:39]` | `[27:36]` |
| thumb1/2/3 | 13–15 | `[39:48]` | `[36:45]` |

The `hand_pose` slice for joint `i`, counting from 1 to 15, is

```text
hand_pose[3*(i-1) : 3*(i-1)+3]
```


In [3]:
def hand_pose_slice(joint_index):
    if joint_index == 0:
        raise ValueError("the wrist is global_orient, not a slice of hand_pose")
    start = 3 * (joint_index - 1)
    return start, start + 3

for index, name in enumerate(sk.JOINT_NAMES):
    if index == 0:
        print(f"{index:02d} {name:<8} global_orient[0:3]")
    else:
        start, stop = hand_pose_slice(index)
        print(f"{index:02d} {name:<8} hand_pose[{start}:{stop}]")

assert hand_pose_slice(2) == (3, 6)     # index PIP
assert hand_pose_slice(11) == (30, 33)  # ring PIP


00 wrist    global_orient[0:3]
01 index1   hand_pose[0:3]
02 index2   hand_pose[3:6]
03 index3   hand_pose[6:9]
04 middle1  hand_pose[9:12]
05 middle2  hand_pose[12:15]
06 middle3  hand_pose[15:18]
07 pinky1   hand_pose[18:21]
08 pinky2   hand_pose[21:24]
09 pinky3   hand_pose[24:27]
10 ring1    hand_pose[27:30]
11 ring2    hand_pose[30:33]
12 ring3    hand_pose[33:36]
13 thumb1   hand_pose[36:39]
14 thumb2   hand_pose[39:42]
15 thumb3   hand_pose[42:45]


The index PIP is joint 2, so it lives in `hand_pose[3:6]`. Those three numbers are always `(rx, ry, rz)`.

## Fingertips are not in the tree

There is no joint 16. Fingertips are mesh vertices:

| Finger | Vertex id |
|---|---|
| thumb | 744 |
| index | 320 |
| middle | 443 |
| ring | 554 |
| pinky | 671 |

Joint 15 is the thumb IP, one bone short of the tip. On this flat right-hand template, joint 15 is about 35 mm from vertex 744. Matching joint 15 to the thumb tip matches a different point. Chapter 10 measures that distance.

## Check

Without looking at the table, compute the slice of the ring PIP. The next cell prints the answer so you can compare.


In [6]:
print("pinky MCP index", sk.JOINT_NAMES.index("pinky1"))
print("ring PIP hand_pose slice", hand_pose_slice(sk.JOINT_NAMES.index("ring2")))
print("thumb tip vertex", sk.TIP_VERTEX_IDS["thumb"])


pinky MCP index 7
ring PIP hand_pose slice (30, 33)
thumb tip vertex 744
